# 05 - Error analysis and explanations
Where does the model fail? Which segments are poorly served? What drives individual predictions (SHAP, associative not causal)?

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from credit_risk.config import load_config
from credit_risk.data.loader import load_dataset, dataset_status_report
cfg = load_config()
print(dataset_status_report(cfg))
# demo=True is an EXPLICIT opt-in: if the real Home Credit files are missing, SYNTHETIC data is generated.
df, DATA_SOURCE = load_dataset(cfg, demo=True)
print("\nDATA SOURCE:", DATA_SOURCE, df.shape)
if DATA_SOURCE != "home_credit":
    print("WARNING: SYNTHETIC data - numbers are only a code-path demonstration, not real results.")

In [ ]:
import tempfile
from credit_risk.config import apply_lightweight
from credit_risk.models.train import run_training
from credit_risk.models.predict import ModelBundle
from credit_risk.features.preprocessing import split_data
cfg_lw = apply_lightweight(cfg); cfg_lw['mlflow']['enabled'] = False
d = df.sample(min(len(df), 30000), random_state=0)
tmp = Path(tempfile.mkdtemp())
result = run_training(cfg_lw, d, DATA_SOURCE, tmp/'a', tmp/'r', mlflow_enabled=False, run_shap=True)
bundle = ModelBundle.load(tmp/'a'/'model_bundle.joblib')
*_, X_te, _, _, y_te = split_data(d,'TARGET',cfg_lw['split']['test_size'],cfg_lw['split']['val_size'],cfg_lw['project']['seed'])
p = bundle.predict_proba(X_te)

In [ ]:
dec = pd.qcut(p, 10, labels=False, duplicates='drop')
lift = pd.DataFrame({'decile':dec,'y':y_te.values,'p':p}).groupby('decile').agg(applicants=('y','size'), observed=('y','mean'), predicted=('p','mean')).round(4)
lift

In [ ]:
thr = cfg['decision']['policies']['balanced']['flag_threshold']
flag = p >= thr
fn = (~flag) & (y_te.values==1); fp = flag & (y_te.values==0)
print(f'threshold {thr}: false negatives={fn.sum()}, false positives={fp.sum()}, flagged={flag.mean():.1%}')

In [ ]:
seg = X_te.assign(y=y_te.values, flag=flag, fn=fn)
for col in ['NAME_INCOME_TYPE','NAME_EDUCATION_TYPE']:
    print(seg.groupby(col).agg(n=('y','size'), default_rate=('y','mean'), flagged=('flag','mean'), missed_defaults=('fn','sum')).round(3), '\n')

Segment gaps in flag rates or missed defaults are *descriptive* signals to investigate; a fairness assessment needs legally appropriate attributes, outcome definitions and statistical testing (see `reports/limitations.md`).

In [ ]:
from credit_risk.explainability.shap_analysis import explain_instance
i_fn = int(np.flatnonzero(fn)[0]) if fn.any() else 0
print('Missed defaulter, predicted PD =', round(float(p[i_fn]),4))
explain_instance(bundle, X_te.iloc[[i_fn]], top_k=8)

SHAP values attribute the *model's* output to inputs relative to a background sample. They do not establish causal effects, can split credit among correlated features, and may expose proxies of sensitive attributes.